# Aviation Accidents Analysis — Data Cleaning

This notebook loads, inspects, and cleans the NTSB aviation accidents dataset (1948-2023). The client (an airline/airplane insurer) needs aircraft safety analysis; we filter to **professional builds from 1983 onwards** and derive key safety metrics.

### Library Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

## 1. Data Loading and Inspection

In [ ]:
# Load with latin-1 encoding (dataset contains non-UTF-8 bytes)
df = pd.read_csv('AviationData.csv', low_memory=False, encoding='latin1')
print('Shape:', df.shape)
df.head(3)

In [ ]:
# Data types
df.dtypes

In [ ]:
# Missing value percentages
nan_pct = (df.isnull().mean() * 100).round(1).sort_values(ascending=False)
print(nan_pct.to_string())

In [ ]:
# Summary statistics for numeric columns
df.describe()

## 2. Data Cleaning

### 2.1 Filtering Aircraft and Events

We apply the following filters:
1. **Date ≥ 1983**: Client wants makes/models that could still be active (40-year lifetime ceiling).
2. **Aircraft.Category**: Retain only Airplanes (drop Helicopters, Balloons, Gliders, etc.); where category is NaN we defer to Amateur.Built filter.
3. **Amateur.Built == 'No'**: Professional builds only.
4. **Investigation.Type == 'Accident'**: Remove incidents and other investigation types.

In [ ]:
# 1. Parse dates and filter 1983+
df['Event.Date'] = pd.to_datetime(df['Event.Date'], errors='coerce')
df = df[df['Event.Date'].dt.year >= 1983].copy()
print('After date filter:', df.shape)

# 2. Remove non-airplane categories
cat_exclude = ['Helicopter','Glider','Balloon','Gyrocraft','Weight-Shift',
               'Powered Parachute','Ultralight','Rocket']
df = df[~df['Aircraft.Category'].isin(cat_exclude)].copy()
print('After category filter:', df.shape)

# 3. Amateur Built filter
df = df[df['Amateur.Built'] == 'No'].copy()
print('After amateur filter:', df.shape)

# 4. Accidents only
df = df[df['Investigation.Type'] == 'Accident'].copy()
print('After accident filter:', df.shape)

### 2.2 Constructing the Fatal/Serious Injury Metric

**Approach:** We estimate total passengers aboard as the sum of all injury categories. The safety metric is:

```
Fatal_Serious_Frac = (Total.Fatal.Injuries + Total.Serious.Injuries) / Total.Aboard
```

**Cleaning assumptions:**
- Injury counts with NaN are imputed to **0** (no injury recorded → assume none).
- Rows where `Total.Aboard == 0` are set to NaN to avoid division-by-zero (likely missing data, not real zero-passenger flights).
- The fraction is clipped to [0, 1] to handle any data inconsistencies.

In [ ]:
injury_cols = ['Total.Fatal.Injuries','Total.Serious.Injuries',
               'Total.Minor.Injuries','Total.Uninjured']
for c in injury_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0)

# Total passengers aboard (proxy for capacity)
df['Total.Aboard'] = df[injury_cols].sum(axis=1)
df.loc[df['Total.Aboard'] == 0, 'Total.Aboard'] = np.nan  # mask true unknowns

# Injury fraction
df['Fatal_Serious_Frac'] = (
    (df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']) / df['Total.Aboard']
).clip(0, 1)

print('Fatal_Serious_Frac stats:')
print(df['Fatal_Serious_Frac'].describe())

### 2.3 Aircraft.Damage → Destroyed Flag

We convert the damage column into a binary indicator: **1 = Destroyed, 0 = Not destroyed**. Rows with 'Unknown' or NaN damage are masked to NaN so they don't bias averages.

In [ ]:
df['Aircraft.damage'] = df['Aircraft.damage'].str.strip().str.title()
print(df['Aircraft.damage'].value_counts(dropna=False))

# Binary destroyed flag
df['Destroyed'] = (df['Aircraft.damage'] == 'Destroyed').astype(float)
df.loc[df['Aircraft.damage'] == 'Unknown', 'Destroyed'] = np.nan
df.loc[df['Aircraft.damage'].isna(), 'Destroyed'] = np.nan
print('\nDestroyed flag distribution:')
print(df['Destroyed'].value_counts(dropna=False))

### 2.4 Cleaning the Make Column

**Tasks identified:**
1. Strip whitespace and standardise to UPPERCASE.
2. Consolidate common aliases (e.g. 'CESSNA AIRCRAFT' → 'CESSNA').
3. Retain only Makes with **≥ 50 records** to ensure statistical reliability.

In [ ]:
df['Make'] = df['Make'].str.strip().str.upper()

# Alias map for common duplicates
make_map = {
    'CESSNA AIRCRAFT': 'CESSNA', 'CESSNA AIRCRAFT CO': 'CESSNA',
    'BEECHCRAFT': 'BEECH', 'BEECH AIRCRAFT': 'BEECH',
    'PIPER AIRCRAFT': 'PIPER', 'PIPER AIRCRAFT INC': 'PIPER',
    'BOEING COMMERCIAL AIRPLANES': 'BOEING',
    'AIRBUS INDUSTRIE': 'AIRBUS',
    'RAYTHEON AIRCRAFT': 'RAYTHEON', 'RAYTHEON AIRCRAFT CO': 'RAYTHEON',
    'MOONEY AIRCRAFT': 'MOONEY', 'MOONEY AIRCRAFT CORP': 'MOONEY',
    'GRUMMAN AMERICAN': 'GRUMMAN', 'AMERICAN GRUMMAN': 'GRUMMAN',
}
df['Make'] = df['Make'].replace(make_map)

make_counts = df['Make'].value_counts()
print(f'Makes before threshold filter: {make_counts.shape[0]}')
valid_makes = make_counts[make_counts >= 50].index
df = df[df['Make'].isin(valid_makes)].copy()
print(f'Makes after threshold (≥50): {df["Make"].nunique()}')  
print(df.shape)

### 2.5 Model Column

- Drop rows with NaN Model.
- Model labels are **not unique across makes** (e.g. several makes have a '150' model), so we create `Make_Model` as a unique identifier.

In [ ]:
df = df.dropna(subset=['Model']).copy()
df['Model'] = df['Model'].str.strip().str.upper()

# Verify non-uniqueness
print('Unique models:', df['Model'].nunique())
print('Unique Make_Model combos:', (df['Make'] + ' | ' + df['Model']).nunique())

# Create unique identifier
df['Make_Model'] = df['Make'] + ' | ' + df['Model']
df['Make_Model'].head()

### 2.6 Other Contextual Columns

We clean: Engine.Type, Weather.Condition, Number.of.Engines, Purpose.of.flight, Broad.phase.of.flight.

In [ ]:
# Engine.Type — standardise title case, keep known values only
valid_engines = ['Reciprocating','Turbo Prop','Turbo Fan','Turbo Jet',
                 'Turbo Shaft','Electric','Geared Turbofan']
df['Engine.Type'] = df['Engine.Type'].str.strip().str.title()
df['Engine.Type'] = df['Engine.Type'].where(df['Engine.Type'].isin(valid_engines))

print('Engine.Type:\n', df['Engine.Type'].value_counts(dropna=False).head(10))

# Weather.Condition — keep VMC / IMC only
df['Weather.Condition'] = df['Weather.Condition'].str.strip().str.upper()
df.loc[~df['Weather.Condition'].isin(['VMC','IMC']), 'Weather.Condition'] = np.nan
print('\nWeather.Condition:\n', df['Weather.Condition'].value_counts(dropna=False))

# Number.of.Engines — coerce to numeric
df['Number.of.Engines'] = pd.to_numeric(df['Number.of.Engines'], errors='coerce')

# Purpose.of.flight & Phase — title case
df['Purpose.of.flight'] = df['Purpose.of.flight'].str.strip().str.title()
df['Broad.phase.of.flight'] = df['Broad.phase.of.flight'].str.strip().str.title()

print('\nPhase of flight:\n', df['Broad.phase.of.flight'].value_counts(dropna=False).head(10))

### 2.7 Drop High-NaN Columns (> 70% missing)

In [ ]:
thresh = 0.70
nan_frac = df.isnull().mean()
cols_to_drop = nan_frac[nan_frac > thresh].index.tolist()
print('Dropping columns:', cols_to_drop)
df = df.drop(columns=cols_to_drop)
print('Final shape:', df.shape)

### 2.8 Save Cleaned Data

In [ ]:
df.to_csv('aviation_cleaned.csv', index=False)
print('Saved aviation_cleaned.csv')
print(df.shape)
df.head()